# 第 24 章：信用风险与违约建模

本章的主要算例，从信用三角到负基差。

| 节 | 复现什么 | 例 |
|---|---|---|
| 1 | 信用三角对连续支付的保费是精确的；所谓“误差”其实是应计保费 | 24.1 |
| 2 | 由利差曲线自举出违约强度曲线 | 24.2 |
| 3 | 违约强度与回收率无法分别识别 | 24.3 |
| 4 | 由股价算出违约距离（Merton） | 24.4 |
| 5 | 信用利差的大部分并不是预期损失 | 24.5 |
| 6 | 降级是通往违约的路 | 24.6 |
| 7 | 相关性把概率推向两个尾部 | 24.7 |
| 8 | 负基差为什么不是套利 | 24.12 |

全章都是确定性的（没有任何模拟），因此每一个数字都应当精确复现到正文所印的位数。
凡正文只印一位小数而计算给出更多位的地方，两者都会印出来。

用 NumPy 与 SciPy。

表头保留英文：中文字符在等宽终端里是双宽的，而 Python 的 `%-44s` 按字符数计。

In [1]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm, binom
from scipy.optimize import brentq, fsolve

N = norm.cdf
R, RATE, FREQ = 0.40, 0.03, 4          # 回收率、贴现率、按季付费


def show(label, got, printed, tol=6e-4):
    """把算出来的值印在本章所印的值旁边。"""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %12.4f   book %10.4f   %s" % (label, got, printed, flag))


def cds_legs(lam, T=5.0, R=R, r=RATE, freq=FREQ, premium="quarterly", steps=200):
    """常数违约强度下的风险年金与保护端。

    premium = "continuous"  保费连续支付，两端共用同一个积分
              "quarterly"   保费在季末支付，违约时付清应计保费
              "no-accrual"  保费在季末支付，违约时不付任何应计
    在季末支付的保费比连续支付的保费多贴现半个季度；正是这一点把按季合约与
    三角关系分开了大约 r*dt/2。
    """
    n, dt = int(round(T * freq)), 1.0 / freq
    h = dt / steps
    ann = prot = 0.0
    for i in range(n):
        t0 = i * dt
        if premium != "continuous":
            ann += dt * exp(-r * (t0 + dt)) * exp(-lam * (t0 + dt))
        for k in range(steps):
            t = t0 + (k + 0.5) * h
            dens = lam * exp(-lam * t) * h           # 该小段上的违约密度
            prot += exp(-r * t) * dens
            if premium == "continuous":
                ann += exp(-r * t) * exp(-lam * t) * h
            elif premium == "quarterly":
                ann += (t - t0) * exp(-r * t) * dens
    return ann, (1 - R) * prot


print("一份五年期 CDS，按季付费，r = %.0f%%，回收率 %.0f%%"
      % (100 * RATE, 100 * R))

一份五年期 CDS，按季付费，r = 3%，回收率 40%


## 1. 信用三角对连续支付的保费是精确的：例 24.1

保费连续支付时，$s = \lambda(1-R)$ 是精确的：生存概率与贴现因子在两端同时出现并互相消掉，
**只有 $(1-R)$ 留下来**。真实合约按季支付，而在季末支付的保费比连续支付的保费多贴现半个季度，这使平价
利差在每个违约强度下都比三角关系高出约 $r\,\Delta/2 \approx 0.4\%$。再去掉违约时的应计保费，
会产生第二个随 $\lambda$ 增长的差距。这就是教科书说法所引的“近似误差”，它是一项
合约特征的价值，而不是三角关系的误差。

In [2]:
print("%8s %14s %22s %20s" % ("lambda", "continuous", "quarterly, accrual", "quarterly, none"))
for lam in (0.01, 0.03, 0.06, 0.10, 0.20):
    rows = [1e4 * p / a for a, p in (cds_legs(lam, premium=m)
                                    for m in ("continuous", "quarterly", "no-accrual"))]
    print("%7.0f%% %14.2f %22.2f %20.2f" % (100 * lam, rows[0], rows[1], rows[2]))
print()
a_c, p_c = cds_legs(0.20, premium="continuous")
a_q, p_q = cds_legs(0.20, premium="quarterly")
a_n, p_n = cds_legs(0.20, premium="no-accrual")
s_c, s_q, s_n = 1e4 * p_c / a_c, 1e4 * p_q / a_q, 1e4 * p_n / a_n
print("连续支付的保费：精确等于 lambda(1-R) = %.1f bp" % (1e4 * 0.20 * (1 - R)))
print("按季支付的时间差值利差的 %.2f%%；再去掉应计，又值 %.1f%%"
      % (100 * (s_q - s_c) / s_c, 100 * (s_n - s_q) / s_q))

  lambda     continuous     quarterly, accrual      quarterly, none
      1%          60.00                  60.23                60.30
      3%         180.00                 180.68               181.36
      6%         360.00                 361.35               364.08
     10%         600.00                 602.25               609.86
     20%        1200.00                1204.47              1235.17

连续支付的保费：精确等于 lambda(1-R) = 1200.0 bp
按季支付的时间差值利差的 0.37%；再去掉应计，又值 2.5%


## 2. 从利差曲线到违约强度曲线：例 24.2

利差是合约整个存续期内的*平均*，因此当曲线上行时，
靠后那一段的违约强度必须高于这个平均，正如远期利率高于互换利率。
把信用三角直接套到每一个报价利差上，对第一段是对的，之后就越来越错。

In [3]:
QUOTES = [(1.0, 0.0080), (3.0, 0.0130), (5.0, 0.0170), (7.0, 0.0190), (10.0, 0.0200)]


def survival(t, knots, lams):
    """分段常数违约强度：Q(t) = exp(-lambda 的积分)。"""
    acc, prev = 0.0, 0.0
    for kt, lam in zip(knots, lams):
        if t <= kt:
            return exp(-(acc + lam * (t - prev)))
        acc += lam * (kt - prev)
        prev = kt
    return exp(-(acc + lams[-1] * (t - prev)))


def par_spread(knots, lams, T, R=R, r=RATE, freq=FREQ):
    n, dt = int(round(T * freq)), 1.0 / freq
    ann = prot = 0.0
    for i in range(1, n + 1):
        t, t_prev = i * dt, (i - 1) * dt
        Q, Q_prev = survival(t, knots, lams), survival(t_prev, knots, lams)
        df = exp(-r * t)
        ann += dt * df * Q + 0.5 * dt * df * (Q_prev - Q)
        prot += df * (Q_prev - Q)
    return (1 - R) * prot / ann


knots, lams = [], []
for Tq, s in QUOTES:                          # 逐段自举
    knots.append(Tq)
    lams.append(0.02)
    lams[-1] = brentq(lambda x: (lams.__setitem__(-1, x),
                                 par_spread(knots, lams, Tq) - s)[1], 1e-8, 3.0)

print("%10s %12s %22s %22s" % ("maturity", "spread (bp)", "bootstrapped lambda", "triangle guess"))
prev = 0.0
for (Tq, s), lam in zip(QUOTES, lams):
    print("%9.0fy %12.0f %19.3f%% (%.0f-%.0fy) %19.3f%%"
          % (Tq, 1e4 * s, 100 * lam, prev, Tq, 100 * s / (1 - R)))
    prev = Tq

print()
print("累计违约概率")
for Tq in (1, 3, 5, 7, 10):
    print("  %2dy   %6.2f%%" % (Tq, 100 * (1 - survival(Tq, knots, lams))))
print()
print("在 3--5 年那一段，与三角猜测之比为 %.2f"
      % (lams[2] / (0.0170 / (1 - R))))

  maturity  spread (bp)    bootstrapped lambda         triangle guess
        1y           80               1.333% (0-1y)               1.333%
        3y          130               2.617% (1-3y)               2.167%
        5y          170               3.998% (3-5y)               2.833%
        7y          190               4.222% (5-7y)               3.167%
       10y          200               3.885% (7-10y)               3.333%

累计违约概率
   1y     1.32%
   3y     6.36%
   5y    13.55%
   7y    20.55%
  10y    29.29%

在 3--5 年那一段，与三角猜测之比为 1.41


## 3. 违约强度与回收率无法分别识别：例 24.3

市场观察到的是**乘积** $\lambda(1-R)$，仅此而已。
回收率假设不是从利差里提取出来的；它是被强加上去的。
下面四行给这份 CDS 的定价完全相同，
而对只取决于 $\lambda$ 的数字违约互换，它们给出的价格相差三倍以上。

In [4]:
S_5Y = 0.0170
print("%10s %16s %26s" % ("recovery", "implied lambda", "5-year default probability"))
for Rx in (0.20, 0.40, 0.60, 0.80):
    lam = S_5Y / (1 - Rx)
    print("%9.0f%% %15.3f%% %25.2f%%" % (100 * Rx, 100 * lam, 100 * (1 - exp(-5 * lam))))

print()
for Rx in (0.20, 0.40, 0.60, 0.80):
    lam = S_5Y / (1 - Rx)
    ann, prot = cds_legs(lam, R=Rx, premium="continuous")            # 每一行都用它自己的回收率定价
    print("  R=%2.0f%%   CDS 重新定价为 %.1f bp   lambda %.3f%%   五年违约 %.2f%%"
          % (100 * Rx, 1e4 * prot / ann, 100 * lam, 100 * (1 - exp(-5 * lam))))
print()
print("四行给这份 CDS 的定价完全相同，这正是要点")
print("数字违约互换只取决于 lambda：各行之间相差 %.1f 倍"
      % ((S_5Y / 0.2) / (S_5Y / 0.8)))
print("而按它实际据以支付的违约概率算，这个比是 %.1f"
      % ((1 - exp(-5 * S_5Y / 0.2)) / (1 - exp(-5 * S_5Y / 0.8))))

  recovery   implied lambda 5-year default probability
       20%           2.125%                     10.08%
       40%           2.833%                     13.21%
       60%           4.250%                     19.14%
       80%           8.500%                     34.62%

  R=20%   CDS 重新定价为 170.0 bp   lambda 2.125%   五年违约 10.08%
  R=40%   CDS 重新定价为 170.0 bp   lambda 2.833%   五年违约 13.21%
  R=60%   CDS 重新定价为 170.0 bp   lambda 4.250%   五年违约 19.14%
  R=80%   CDS 重新定价为 170.0 bp   lambda 8.500%   五年违约 34.62%

四行给这份 CDS 的定价完全相同，这正是要点
数字违约互换只取决于 lambda：各行之间相差 4.0 倍
而按它实际据以支付的违约概率算，这个比是 3.4


## 4. 由股价算出违约距离：例 24.4

股权是以公司资产为标的的看涨期权，因此两个方程、两个未知数，
就能从可观察量里还原出资产价值与资产波动率。
**股权波动率约为资产波动率的两倍**，这个倍数是股权对公司价值的弹性 $N(d_1)V/E$，
即杠杆 $V/E = 2.40$ 按看涨期权的 delta 缩小后的值。

还要注意 Merton 模型的隐含回收率偏高：违约时债权人无偿接管公司的全部资产，
这些资产平均仍值债务面值的约 70%，而模型中没有破产成本来压低这个数。

In [5]:
E, SIG_E, D_FACE, T_M, R_M = 40.0, 0.55, 80.0, 5.0, 0.05


def merton_system(p):
    V, sig_V = p
    d1 = (log(V / D_FACE) + (R_M + 0.5 * sig_V ** 2) * T_M) / (sig_V * sqrt(T_M))
    d2 = d1 - sig_V * sqrt(T_M)
    equity = V * N(d1) - D_FACE * exp(-R_M * T_M) * N(d2)
    return [equity - E, V * N(d1) * sig_V / equity - SIG_E]


V, SIG_V = fsolve(merton_system, [E + D_FACE * exp(-R_M * T_M), SIG_E * E / (E + D_FACE)])
d1 = (log(V / D_FACE) + (R_M + 0.5 * SIG_V ** 2) * T_M) / (SIG_V * sqrt(T_M))
d2 = d1 - SIG_V * sqrt(T_M)
debt = V - E
riskless = D_FACE * exp(-R_M * T_M)
spread = -log(debt / D_FACE) / T_M - R_M

show("implied asset value", V, 95.87, tol=3e-4)
show("implied asset volatility, %", 100 * SIG_V, 27.2, tol=3e-3)
show("distance to default", d2, 0.41, tol=2e-2)
show("risk-neutral 5y default probability, %", 100 * N(-d2), 34.26, tol=2e-3)
show("implied credit spread, bp", 1e4 * spread, 218, tol=5e-3)
print()
show("debt value", debt, 55.87, tol=3e-4)
show("riskless PV of debt", riskless, 62.30, tol=3e-4)
print("预期损失 %.1f%%，而违约概率为 %.0f%%"
      % (100 * (1 - debt / riskless), 100 * N(-d2)))
print("隐含回收率接近 %.0f%%"
      % (100 * (1 - (1 - debt / riskless) / N(-d2))))
print("股权波动率／资产波动率 = %.2f，即弹性 N(d1) V/E" % (SIG_E / SIG_V))

implied asset value                                 95.8691   book    95.8700   ok
implied asset volatility, %                         27.1739   book    27.2000   ok
distance to default                                  0.4054   book     0.4100   ok
risk-neutral 5y default probability, %              34.2580   book    34.2600   ok
implied credit spread, bp                          218.0305   book   218.0000   ok

debt value                                          55.8691   book    55.8700   ok
riskless PV of debt                                 62.3041   book    62.3000   ok
预期损失 10.3%，而违约概率为 34%
隐含回收率接近 70%
股权波动率／资产波动率 = 2.02，即弹性 N(d1) V/E


## 5. 利差的大部分并不是预期损失：例 24.5

**而且高评级主体的倍数最大。**
AA 或 A 主体的风险中性违约概率约为历史违约率的十六倍，B 级主体则不到两倍半；
AAA 较低，只是因为其小样本的历史违约率高于 AA。
无论这部分超额补偿的是什么，它都不随违约风险等比例变化。

用风险中性概率去定价与对冲，用历史概率去预测损失，而且绝不混用。

In [6]:
# five-year average cumulative default rates, S&P Global Ratings 2024 annual default study, 1981-2024
RATINGS = [("AAA", 40, 0.0034), ("AA", 55, 0.0028), ("A", 80, 0.0039),
           ("BBB", 130, 0.0136), ("BB", 300, 0.0575), ("B", 550, 0.1560)]

print("%10s %10s %14s %16s %10s" % ("rating", "spread", "historical", "risk-neutral", "ratio"))
for name, s_bp, hist in RATINGS:
    rn = 1 - exp(-5 * (s_bp / 1e4) / (1 - R))
    print("%10s %8d bp %13.2f%% %15.2f%% %9.1fx" % (name, s_bp, 100 * hist, 100 * rn, rn / hist))

print()
lam_hist = -log(1 - 0.0136) / 5
el = 1e4 * lam_hist * (1 - R)
show("BBB expected loss as a spread, bp", el, 16.4, tol=5e-3)
show("BBB residual, bp", 130 - el, 113.6, tol=5e-3)
print("因此 BBB 利差中有 %.0f%% 不是预期损失" % (100 * (130 - el) / 130))

    rating     spread     historical     risk-neutral      ratio
       AAA       40 bp          0.34%            3.28%       9.6x
        AA       55 bp          0.28%            4.48%      16.0x
         A       80 bp          0.39%            6.45%      16.5x
       BBB      130 bp          1.36%           10.27%       7.5x
        BB      300 bp          5.75%           22.12%       3.8x
         B      550 bp         15.60%           36.77%       2.4x

BBB expected loss as a spread, bp                   16.4320   book    16.4000   ok
BBB residual, bp                                   113.5680   book   113.6000   ok
因此 BBB 利差中有 87% 不是预期损失


## 6. 降级是通往违约的路：例 24.6

假定各年独立、把一年期 BBB 违约率逐年复合，得到 1.39\%。
转移矩阵给出的是 **2.89\%**，为前者的 2.08 倍。
矩阵给得更多，是因为一个到第四年尚未违约的 BBB 级主体，
那时多半已经是 BB 或 B 了，面对的违约强度远高于它起步时的 0.28\%。

In [7]:
GRADES = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "D"]
M = np.array([
    [90.09, 9.01, 0.60, 0.10, 0.10, 0.10, 0.00, 0.00],
    [0.60, 90.59, 7.91, 0.60, 0.10, 0.20, 0.00, 0.00],
    [0.05, 2.10, 91.40, 5.50, 0.60, 0.20, 0.10, 0.05],
    [0.02, 0.20, 4.20, 89.90, 4.30, 0.90, 0.20, 0.28],
    [0.02, 0.06, 0.30, 5.60, 83.00, 8.50, 1.00, 1.52],
    [0.00, 0.05, 0.20, 0.40, 5.80, 82.00, 5.10, 6.45],
    [0.00, 0.00, 0.30, 0.60, 1.50, 12.00, 55.00, 30.60],
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 100.0]]) / 100.0

M5 = np.linalg.matrix_power(M, 5)              # D 是吸收态，因此矩阵幂就是五年转移矩阵
row = M5[GRADES.index("BBB")]
print("一个 BBB 主体五年后的去向")
print("  " + "".join("%8s" % g for g in GRADES))
print("  " + "".join("%7.2f%%" % (100 * x) for x in row))
print()
print("仍为投资级 %.1f%%，跌为投机级 %.1f%%，已违约 %.2f%%"
      % (100 * row[:4].sum(), 100 * row[4:7].sum(), 100 * row[7]))

naive = 1 - (1 - 0.0028) ** 5
show("naive compounding of 0.28%, %", 100 * naive, 1.39, tol=5e-3)
show("transition matrix, %", 100 * row[7], 2.89, tol=5e-3)
show("ratio", row[7] / naive, 2.08, tol=1e-2)

一个 BBB 主体五年后的去向
       AAA      AA       A     BBB      BB       B     CCC       D
     0.10%   1.37%  14.64%  62.17%  12.90%   5.06%   0.87%   2.89%

仍为投资级 78.3%，跌为投机级 18.8%，已违约 2.89%
naive compounding of 0.28%, %                        1.3922   book     1.3900   ok
transition matrix, %                                 2.8899   book     2.8900   ok
ratio                                                2.0758   book     2.0800   ok


## 7. 相关性把概率推向两个尾部：例 24.7

**预期损失完全不变。**
相关性做的事，是把概率从中间拿走、放到*两个*端点上：
在 $\rho = 0.6$ 时，有 53\% 的概率一个都不违约，
也有 8\% 的概率至少五分之一的组合违约。

优先档几乎只是一项相关性赌注，原因正是这种双尾行为，
而这就是第 26 章的全部主题。

In [8]:
def portfolio_distribution(rho, p=0.05, n=100, nodes=800):
    """单因子高斯 copula：把条件二项分布对该因子积分。"""
    x, wq = np.polynomial.legendre.leggauss(nodes)
    z = 8.0 * x                                 # 该因子，在 8 倍标准差处截断
    w = 8.0 * wq * norm.pdf(z)
    c = norm.ppf(p)
    pz = N((c - sqrt(rho) * z) / sqrt(1 - rho)) if rho > 0 else np.full_like(z, p)
    pz = np.clip(pz, 1e-15, 1 - 1e-15)
    k = np.arange(n + 1)
    return (binom.pmf(k[None, :], n, pz[:, None]) * w[:, None]).sum(axis=0)


print("一百个主体，每个的违约概率为 5%")
print("%8s %12s %12s %18s %16s" % ("rho", "expected", "std dev", "P(no defaults)", "P(20 or more)"))
k = np.arange(101)
for rho in (0.00, 0.05, 0.15, 0.30, 0.60):
    pmf = portfolio_distribution(rho)
    mean = float((k * pmf).sum())
    sd = sqrt(float((k ** 2 * pmf).sum()) - mean ** 2)
    print("%8.2f %12.2f %12.2f %17.2f%% %15.2f%%"
          % (rho, mean, sd, 100 * pmf[0], 100 * pmf[20:].sum()))

一百个主体，每个的违约概率为 5%
     rho     expected      std dev     P(no defaults)    P(20 or more)


    0.00         5.00         2.18              0.59%            0.00%


    0.05         5.00         3.22              3.13%            0.11%


    0.15         5.00         4.89             10.86%            1.85%


    0.30         5.00         7.12             24.62%            4.91%


    0.60         5.00        11.56             53.02%            7.87%


## 8. 负基差为什么不是套利：例 24.12

买入债券、买入保护、赚取基差，只是这只债券必须融资，
而公司债的回购利率不是无风险利率。**这笔交易是亏钱的**，
而且还要额外占用资产负债表。

因此负基差不是套利；它是资产负债表成本的一个度量，
而且只要融资成本还高于它，它就会一直存在。

In [9]:
bond_spread, cds_spread, funding = 220.0, 170.0, 60.0
basis = cds_spread - bond_spread
show("the basis, bp", basis, -50.0)
show("carry before funding, bp", bond_spread - cds_spread, 50.0)
show("net of funding, bp", bond_spread - cds_spread - funding, -10.0)
print()
print("这笔交易亏 %.0f bp，而且还要额外占用资产负债表"
      % abs(bond_spread - cds_spread - funding))
print("只有当融资成本降到 %.0f bp 以下，它才开始赚钱" % (bond_spread - cds_spread))

the basis, bp                                      -50.0000   book   -50.0000   ok
carry before funding, bp                            50.0000   book    50.0000   ok
net of funding, bp                                 -10.0000   book   -10.0000   ok

这笔交易亏 10 bp，而且还要额外占用资产负债表
只有当融资成本降到 50 bp 以下，它才开始赚钱
